# FLUS-CA Python Library: Single Run, Verification, and Batch Example

This notebook shows how to run the Python implementation of the FLUS-style Cellular Automata (CA) simulation module using a single YAML configuration file.

It includes:

1. Loading the library from the repository.
2. Reading a YAML configuration file.
3. Running one short test simulation.
4. Checking whether the output raster changed compared with the input raster.
5. Summarizing class counts and transitions.
6. Running an optional batch calibration example.

> This notebook does **not** implement the ANN probability module. It assumes the probability raster already exists.

## 1. Setup

This cell makes the local repository importable. It works when the notebook is inside a `notebooks/` folder in the repository.

Expected repository structure:

```text
pyflus-ca/
├── flus_ca/
├── notebooks/
│   └── 01_run_verify_batch_example.ipynb
├── examples/
├── pyproject.toml
├── README.md
└── flus_config_template.yml
```

In [ ]:
from pathlib import Path
import sys

# If this notebook is stored in pyflus-ca/notebooks/, the repository root is one level above.
REPO_ROOT = Path("..").resolve()

# Add repository root to Python path so that `from flus_ca import FLUSCA` works.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from flus_ca import FLUSCA

print("Repository root:", REPO_ROOT)
print("Library imported successfully.")

## 2. Select the YAML configuration file

For real runs, copy `flus_config_template.yml` to a local file such as `flus_config.yml`.

Recommended:

```text
flus_config.yml
```

should be ignored by Git because it may contain local paths and private data locations.

If you only want to inspect the template, use:

```text
../flus_config_template.yml
```

but the template will only run if the raster paths inside it exist on your computer.

In [ ]:
# Preferred local configuration file.
CONFIG_PATH = REPO_ROOT / "flus_config.yml"

# Fallback to the template if the local config does not exist.
# The template is useful for documentation, but may not run unless its raster paths are valid.
if not CONFIG_PATH.exists():
    CONFIG_PATH = REPO_ROOT / "flus_config_template.yml"

print("Using configuration file:")
print(CONFIG_PATH)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        "No YAML configuration file found. Create flus_config.yml or check the path."
    )

## 3. Load and inspect the model configuration

This does not run the simulation yet. It only checks what the model is going to read.

In [ ]:
model = FLUSCA(CONFIG_PATH)

# Print all key paths and parameters.
model.print_inspect()

## 4. Run a short test simulation

This cell runs only a few iterations to confirm that the model works.

For a full simulation, remove or modify:

```python
model.cfg["simulation"]["max_iterations"] = 3
```

and use the value defined in the YAML, for example 500 iterations.

In [ ]:
# Create a safe output folder for notebook tests.
OUTPUT_DIR = REPO_ROOT / "outputs" / "notebook_test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Avoid overwriting the output path defined in the YAML.
model.cfg["rasters"]["output"] = str(OUTPUT_DIR / "sim_test_notebook.tif")

# Save iteration history.
model.cfg["hyperparameters"]["save_history_csv"] = str(OUTPUT_DIR / "history_test_notebook.csv")

# Short test run. Change this to the original value for a full run.
model.cfg["simulation"]["max_iterations"] = 3

# Run and save.
model.run(verbose=True)
model.save()
model.save_history_csv()

print("Simulation completed.")
print("Output raster:", model.cfg["rasters"]["output"])
print("Final class counts:", model.counts.tolist())

## 5. Verify that the output is different from the input

This verification checks:

- raster dimensions;
- number and percentage of changed pixels;
- class counts before and after the simulation;
- difference between simulated final counts and target demand;
- optional transition table;
- optional binary change map.

If `changed_pixels > 0`, the simulation modified the input map.

In [ ]:
import numpy as np
import pandas as pd
import rasterio


def verify_simulation(input_raster, output_raster, future_pixels=None, output_dir=None):
    """
    Compare an input land-use raster and a simulated output raster.
    """
    input_raster = Path(input_raster)
    output_raster = Path(output_raster)

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    with rasterio.open(input_raster) as src:
        arr_in = src.read(1)
        profile = src.profile.copy()

    with rasterio.open(output_raster) as src:
        arr_out = src.read(1)

    print("Input raster shape:", arr_in.shape)
    print("Output raster shape:", arr_out.shape)

    if arr_in.shape != arr_out.shape:
        raise ValueError("Input and output rasters have different dimensions.")

    valid = arr_in > 0
    changed = (arr_in != arr_out) & valid

    n_valid = int(np.sum(valid))
    n_changed = int(np.sum(changed))
    pct_changed = (n_changed / n_valid * 100) if n_valid > 0 else 0

    print("Valid pixels:", n_valid)
    print("Changed pixels:", n_changed)
    print("Changed percentage:", round(pct_changed, 4), "%")

    classes = sorted(
        set(np.unique(arr_in[valid]).tolist()).union(
            set(np.unique(arr_out[arr_out > 0]).tolist())
        )
    )

    rows = []
    for cls in classes:
        initial_count = int(np.sum(arr_in == cls))
        final_count = int(np.sum(arr_out == cls))
        row = {
            "class": int(cls),
            "initial": initial_count,
            "final": final_count,
            "difference": final_count - initial_count,
        }

        if future_pixels is not None and 1 <= int(cls) <= len(future_pixels):
            target = int(future_pixels[int(cls) - 1])
            row["target_demand"] = target
            row["final_minus_target"] = final_count - target

        rows.append(row)

    counts_df = pd.DataFrame(rows)

    print("
Class count comparison:")
    display(counts_df)

    if n_changed > 0:
        transitions_df = pd.crosstab(
            arr_in[changed].ravel(),
            arr_out[changed].ravel(),
            rownames=["from_class"],
            colnames=["to_class"],
        )
        print("
Transition table for changed pixels:")
        display(transitions_df)
    else:
        transitions_df = pd.DataFrame()
        print("
No changed pixels detected.")

    if output_dir is not None:
        counts_path = output_dir / "class_count_comparison.csv"
        transitions_path = output_dir / "transition_table_changed_pixels.csv"
        change_map_path = output_dir / "binary_change_map.tif"

        counts_df.to_csv(counts_path, index=False)
        transitions_df.to_csv(transitions_path)

        profile.update(dtype="uint8", count=1, nodata=0, compress="lzw")
        with rasterio.open(change_map_path, "w", **profile) as dst:
            dst.write(changed.astype("uint8"), 1)

        print("
Saved verification outputs:")
        print("Class count table:", counts_path)
        print("Transition table:", transitions_path)
        print("Binary change map:", change_map_path)

    return {
        "valid_pixels": n_valid,
        "changed_pixels": n_changed,
        "changed_percentage": pct_changed,
        "counts": counts_df,
        "transitions": transitions_df,
    }


verification = verify_simulation(
    input_raster=model.cfg["rasters"]["landuse"],
    output_raster=model.cfg["rasters"]["output"],
    future_pixels=model.cfg["simulation"]["future_pixels"],
    output_dir=OUTPUT_DIR,
)

## 6. Basic interpretation of the verification

Use this rule:

- `changed_pixels = 0`: the model did not modify the land-use map.
- `changed_pixels > 0`: the simulation changed at least some cells.
- The `difference` column shows how many pixels each class gained or lost.
- The `final_minus_target` column shows how far each class is from the demand target.

For short test runs, it is normal that the model does not fully reach the target demand.

In [ ]:
changed_pixels = verification["changed_pixels"]
changed_percentage = verification["changed_percentage"]

if changed_pixels > 0:
    print(f"The simulation modified the map: {changed_pixels:,} pixels changed.")
    print(f"Changed area fraction: {changed_percentage:.4f}% of valid pixels.")
else:
    print("No changed pixels detected. Check the YAML parameters, demand, cost matrix, and restrictions.")

## 7. Optional batch example

The next cells show how to run several simulations by changing selected neighborhood weights.

This is useful for calibration. For example, you can test whether different neighborhood weights produce more realistic spatial patterns.

By default, the batch is disabled to avoid launching many heavy simulations by accident.

In [ ]:
import copy
import itertools

RUN_BATCH = False  # Change to True only when you want to run the batch.

batch_values = {
    "Landuse3": [0.1, 0.5, 1.0],
    "Landuse7": [0.1, 0.5, 1.0],
    "Landuse8": [0.1, 0.5, 1.0],
}

# Use a small number of iterations for a test batch.
BATCH_MAX_ITERATIONS = 3

BATCH_DIR = REPO_ROOT / "outputs" / "batch_test"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

print("Batch enabled:", RUN_BATCH)
print("Batch output folder:", BATCH_DIR)

In [ ]:
def landuse_name_to_index(name):
    """
    Convert a class name such as 'Landuse3' to a zero-based Python index.
    """
    name = str(name)
    if not name.lower().startswith("landuse"):
        raise ValueError(f"Invalid class name: {name}")
    return int(name.lower().replace("landuse", "")) - 1


if RUN_BATCH:
    keys = list(batch_values.keys())
    values = [batch_values[k] for k in keys]

    batch_summary = []

    for run_id, combo in enumerate(itertools.product(*values), start=1):
        cfg_i = copy.deepcopy(model.cfg)

        weights = list(cfg_i["simulation"]["neighborhood_weights"])
        for class_name, value in zip(keys, combo):
            idx = landuse_name_to_index(class_name)
            weights[idx] = float(value)

        cfg_i["simulation"]["neighborhood_weights"] = weights
        cfg_i["simulation"]["max_iterations"] = BATCH_MAX_ITERATIONS

        run_name = f"batch_{run_id:03d}"
        cfg_i["rasters"]["output"] = str(BATCH_DIR / f"{run_name}.tif")
        cfg_i["hyperparameters"]["save_history_csv"] = str(BATCH_DIR / f"{run_name}_history.csv")

        cfg_i["hyperparameters"]["seed"] = int(cfg_i["hyperparameters"].get("seed", 123)) + run_id

        m = FLUSCA(cfg_i)
        m.run(verbose=False)
        m.save()
        m.save_history_csv()

        v = verify_simulation(
            input_raster=cfg_i["rasters"]["landuse"],
            output_raster=cfg_i["rasters"]["output"],
            future_pixels=cfg_i["simulation"]["future_pixels"],
            output_dir=BATCH_DIR / run_name,
        )

        row = {
            "run_id": run_id,
            "output": cfg_i["rasters"]["output"],
            "changed_pixels": v["changed_pixels"],
            "changed_percentage": v["changed_percentage"],
        }

        for class_name, value in zip(keys, combo):
            row[class_name] = value

        batch_summary.append(row)
        print(f"Finished run {run_id}: {cfg_i['rasters']['output']}")

    batch_summary_df = pd.DataFrame(batch_summary)
    batch_summary_path = BATCH_DIR / "batch_summary.csv"
    batch_summary_df.to_csv(batch_summary_path, index=False)

    print("
Batch completed.")
    print("Summary table:", batch_summary_path)
    display(batch_summary_df)

else:
    print("Batch was not executed. Set RUN_BATCH = True to run it.")

## 8. Recommended files to keep in the repository

Good files to commit:

```text
flus_ca/
examples/
notebooks/
README.md
pyproject.toml
flus_config_template.yml
.gitignore
```

Files that should usually stay out of GitHub:

```text
*.tif
*.tiff
*.csv
*.log
flus_config.yml
local_config.yml
```

The real raster data and local YAML configuration should remain local unless you intentionally publish a small sample dataset.